## Fuente 1: FRED — Federal Reserve Bank of St. Louis
**URL:** https://fred.stlouisfed.org  
**Acceso:** API gratuita (requiere registro de 2 minutos para obtener API key)  
**Actualización:** Diaria / Mensual según la serie  

### ¿Qué es?
FRED (Federal Reserve Economic Data) es el repositorio de datos económicos y financieros 
del Banco de la Reserva Federal de St. Louis. Contiene más de 800,000 series de datos de 
fuentes oficiales como el FMI, el Banco Mundial, la BLS y agencias gubernamentales de EE.UU. 
Es considerado el estándar de referencia para datos macroeconómicos públicos.

### ¿Qué estamos extrayendo?
De FRED extraemos 4 series clave para el proyecto:

| Serie ID       | Variable                        | Uso en el modelo                        |
|----------------|---------------------------------|-----------------------------------------|
| PWHEAMTUSDQ    | Precio global del trigo (IMF)   | Variable de referencia de precio $/TM   |
| CPIAUCSL       | Índice de precios USA (CPI)     | Deflactar precios nominales a reales    |
| DTWEXBGS       | Tipo de cambio USD ponderado    | Feature del modelo DS-1                 |
| BAMLH0A0HYM2   | Proxy de condiciones crediticias| Correlato del Baltic Dry Index          |

### Período y granularidad
- Precio trigo: trimestral, desde 2003
- CPI: mensual, desde 1947
- Tipo de cambio: diario, desde 2006

In [3]:
from fredapi import Fred

fred = Fred(api_key='2f55df367c8ec1c44c28ed52b959e6a5')

In [8]:
from fredapi import Fred

fred = Fred(api_key='2f55df367c8ec1c44c28ed52b959e6a5')

# Test
gdp = fred.get_series('GDP')
print(gdp.tail())

2024-10-01    29825.182
2025-01-01    30042.113
2025-04-01    30485.729
2025-07-01    31098.027
2025-10-01    31422.526
dtype: float64


In [1]:
# Instalación de dependencias
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "fredapi"])

0

In [10]:
import pandas as pd
from fredapi import Fred

fred = Fred(api_key='2f55df367c8ec1c44c28ed52b959e6a5')

# Series a descargar
series = {
    'wheat_price':  'PWHEAMTUSDQ',   # Precio trigo global $/TM
    'cpi_usa':      'CPIAUCSL',      # CPI para deflactar
    'usd_index':    'DTWEXBGS',      # Tipo de cambio USD
    'bdi':          'BAMLH0A0HYM2',  # High Yield spread (proxy riesgo)
}

dfs = {}
for name, sid in series.items():
    dfs[name] = fred.get_series(sid, observation_start='2000-01-01')
    print(f"✓ {name}: {len(dfs[name])} registros")

# Unir en un DataFrame
df = pd.DataFrame(dfs)
df.index = pd.to_datetime(df.index)
df.index.name = 'date'

# Preview
print(df.tail(5))
print(f"\nShape: {df.shape}")
print(f"Nulos: {df.isnull().sum().to_dict()}")

✓ wheat_price: 104 registros
✓ cpi_usa: 315 registros
✓ usd_index: 5290 registros
✓ bdi: 6945 registros
            wheat_price  cpi_usa  usd_index   bdi
date                                             
2026-04-07          NaN      NaN   120.3200  3.12
2026-04-08          NaN      NaN   119.0596  2.94
2026-04-09          NaN      NaN   118.8998  2.90
2026-04-10          NaN      NaN   118.8552  2.94
2026-04-13          NaN      NaN        NaN  2.95

Shape: (7037, 4)
Nulos: {'wheat_price': 6933, 'cpi_usa': 6723, 'usd_index': 1955, 'bdi': 175}


## Fuente 2: USDA ERS — Wheat Data
**URL:** https://www.ers.usda.gov/data-products/wheat-data  
**Acceso:** Descarga directa sin registro (archivos Excel públicos)  
**Actualización:** Mensual (junto con el reporte WASDE)  

### ¿Qué es?
El Economic Research Service del Departamento de Agricultura de EE.UU. (USDA ERS) publica 
el dataset más completo de estadísticas de trigo en el mundo. Incluye datos desde 1918 
desagregados por las 5 clases de trigo estadounidense: Hard Red Winter (HRW), Hard Red 
Spring (HRS), Soft Red Winter (SRW), White y Durum. Es la fuente primaria que alimenta 
los reportes WASDE mensuales, la referencia global para el mercado triguero.

### ¿Qué estamos extrayendo?
Extraemos los archivos Excel de balance oferta-demanda, precios y comercio exterior:

| Archivo              | Contenido                                      | Uso                              |
|----------------------|------------------------------------------------|----------------------------------|
| Table01 (All Wheat)  | Balance nacional USA: producción, stocks, uso  | Balance sheet BI Dashboard 1     |
| Table02–06 (By class)| Balance por clase HRW, HRS, SRW, White, Durum  | Análisis por variedad            |
| Table21–25 (Exports) | Exportaciones USA por clase y destino          | Flujos comerciales Dashboard 1   |
| Table31–35 (World)   | Balance mundial de trigo por país              | Feature DS-3, contexto global    |

### Período y granularidad
- Series históricas desde 1918 (producción anual)
- Precios y comercio: mensual desde 1975
- Marketing year: junio–mayo (requiere conversión a año calendario)

In [15]:
import pandas as pd
import requests
import io

headers = {'User-Agent': 'Mozilla/5.0'}

# Un solo archivo con TODAS las tablas (actualizado 18/03/2026)
url = "https://www.ers.usda.gov/media/5706/wheat-data-all-years.xlsx?v=43491"

resp = requests.get(url, timeout=30, headers=headers)
resp.raise_for_status()

# Ver qué hojas contiene
xl = pd.ExcelFile(io.BytesIO(resp.content))
print("Hojas disponibles:")
for i, sheet in enumerate(xl.sheet_names):
    print(f"  {i}: {sheet}")

Hojas disponibles:
  0: Contents
  1: Table01
  2: Table02
  3: Table03
  4: Table04
  5: Table05
  6: Table06
  7: Table07
  8: Table08
  9: Table09
  10: Table10
  11: Table11
  12: Table12
  13: Table13
  14: Table14
  15: Table15
  16: Table16
  17: Table17
  18: Table18
  19: Table19
  20: Table20
  21: Table21
  22: Table22
  23: Table23
  24: Table24
  25: Table25
  26: Table26
  27: Table27
  28: Table28
  29: Table29
  30: Table30
  31: Table31
  32: Table32
  33: Table33
  34: Table34
  35: Table35


## Fuente 3: USDA FAS — Production, Supply and Distribution (PSD)
**URL:** https://apps.fas.usda.gov/psdonline  
**Acceso:** API REST pública + descarga masiva CSV sin registro  
**Actualización:** Mensual (día de publicación del WASDE)  

### ¿Qué es?
La base de datos PSD del Foreign Agricultural Service del USDA es el dataset de referencia 
mundial para producción, oferta y distribución de granos y oleaginosas por país. Cubre 
aproximadamente 190 países con estimados y proyecciones para todos los commodities agrícolas 
principales. Es la misma fuente que utilizan traders, fondos de cobertura y organismos 
internacionales para monitorear el balance mundial de trigo.

### ¿Qué estamos extrayendo?
Filtramos el commodity trigo (código 0410000) para los 6 países exportadores más relevantes 
para Guatemala:

| País          | Por qué importa para Guatemala                        |
|---------------|-------------------------------------------------------|
| United States | Principal proveedor histórico de trigo HRW             |
| Canada        | Segundo proveedor, trigo HRS de alta proteína          |
| Argentina     | Alternativa competitiva en precio para Centroamérica   |
| Australia     | Referencia de precio en el Pacífico                    |
| Russia        | Mayor exportador mundial, afecta precio global         |
| European Union| Referencia de calidad SRW                              |

**Variables extraídas por país:** área sembrada, producción, exportaciones, 
importaciones, consumo doméstico y stocks finales — todo en miles de toneladas métricas.

### Período y granularidad
- Historial desde 1960 hasta proyecciones del año en curso
- Anual por marketing year local de cada país
- Archivo completo: ~500,000 filas (todos los granos); filtrado a trigo: ~50,000 filas

In [16]:
import pandas as pd
import requests
import io

headers = {'User-Agent': 'Mozilla/5.0'}
url = "https://www.ers.usda.gov/media/5706/wheat-data-all-years.xlsx?v=43491"

resp = requests.get(url, timeout=30, headers=headers)
resp.raise_for_status()
content = io.BytesIO(resp.content)

# Tablas clave
tablas = {
    'us_supply_demand': 'Table01',   # Oferta/demanda EE.UU.
    'world_supply':     'Table10',   # Oferta/demanda mundial
    'us_prices':        'Table18',   # Precios domésticos
    'intl_prices':      'Table19',   # Precios internacionales
    'exports':          'Table21',   # Exportaciones EE.UU.
}

dfs = {}
for name, sheet in tablas.items():
    df = pd.read_excel(content, sheet_name=sheet, skiprows=4, header=0)
    df = df.dropna(how='all').dropna(axis=1, how='all')
    dfs[name] = df
    print(f"✓ {name} ({sheet}): {df.shape}")
    print(f"   Columnas: {df.columns.tolist()[:6]}")
    print()

print("Muestra de precios internacionales:")
print(dfs['intl_prices'].head(5))

✓ us_supply_demand (Table01): (713, 7)
   Columnas: ['All wheat', '1868/69', '--', 19.14, 246.272, 12.9]

✓ world_supply (Table10): (44, 12)
   Columnas: ['1986/87', 198, 232.03, 7, 437.03, 53]

✓ us_prices (Table18): (553, 15)
   Columnas: ['All wheat', '1868/69', '--', '--.1', '--.2', '--.3']

✓ intl_prices (Table19): (498, 15)
   Columnas: ['No. 1 dark northern spring (13% protein) Chicago, IL dollars per bushel', '2011/12', 11.23, 9.75, 9.73, 9.84]

✓ exports (Table21): (150, 19)
   Columnas: ['All wheat flour', '1991/92', 9608.752465375, 5536.40273193144, 4214.66517538042, 3985.64695492388]

Muestra de precios internacionales:
  No. 1 dark northern spring (13% protein) Chicago, IL dollars per bushel  \
0  No. 1 dark northern spring (13% protein) Chica...                        
1  No. 1 dark northern spring (13% protein) Chica...                        
2  No. 1 dark northern spring (13% protein) Chica...                        
3  No. 1 dark northern spring (13% protein) Chica...

In [17]:
import pandas as pd
import requests
import zipfile
import io

# Descarga masiva CSV — archivo completo de granos
url = ("https://apps.fas.usda.gov/psdonline/downloads/"
       "psd_grains_pulses_csv.zip")

resp = requests.get(url, timeout=60)

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    print("Archivos en ZIP:", z.namelist())
    df_raw = pd.read_csv(z.open(z.namelist()[0]))

print(f"Total registros descargados: {df_raw.shape}")

# Filtrar solo trigo
df_wheat = df_raw[df_raw['Commodity_Code'] == 410000].copy()
print(f"Registros de trigo: {df_wheat.shape}")

# Países clave para Guatemala
paises = ['United States', 'Canada', 'Argentina',
          'Australia', 'Russia', 'European Union']
df_key = df_wheat[df_wheat['Country_Name'].isin(paises)]

# Atributos más importantes
attrs = ['Area Harvested', 'Production', 'Exports',
         'Ending Stocks', 'Domestic Consumption']
df_final = df_key[df_key['Attribute_Description'].isin(attrs)]

print(f"\nShape final: {df_final.shape}")
print(df_final.head(5).to_string())

# Pivot para análisis
pivot = df_final.pivot_table(
    index=['Country_Name', 'Market_Year'],
    columns='Attribute_Description',
    values='Value'
).reset_index()

print("\nPivot:")
print(pivot.head(10).to_string())

Archivos en ZIP: ['psd_grains_pulses.csv']
Total registros descargados: (572385, 12)
Registros de trigo: (114465, 12)

Shape final: (1650, 12)
        Commodity_Code Commodity_Description Country_Code Country_Name  Market_Year  Calendar_Year  Month  Attribute_ID Attribute_Description  Unit_ID Unit_Description   Value
461880          410000                 Wheat           AR    Argentina         1960           2006      7             4        Area Harvested        4        (1000 HA)  3599.0
461882          410000                 Wheat           AR    Argentina         1960           2006      7           125  Domestic Consumption        8        (1000 MT)  3294.0
461883          410000                 Wheat           AR    Argentina         1960           2006      7           176         Ending Stocks        8        (1000 MT)   764.0
461884          410000                 Wheat           AR    Argentina         1960           2006      7            88               Exports        8   

## Fuente 4: World Bank Commodity Price Data — Pink Sheet
**URL:** https://www.worldbank.org/en/research/commodity-markets  
**Acceso:** Descarga directa de Excel sin registro  
**Actualización:** Mensual  

### ¿Qué es?
El "Pink Sheet" (llamado así porque originalmente se publicaba en papel rosado) es el 
reporte mensual de precios de commodities del Banco Mundial. Contiene precios nominales 
mensuales de más de 70 commodities desde 1960, incluyendo energía, metales y productos 
agrícolas. Es una de las series de precios más largas y consistentes disponibles 
públicamente, ampliamente usada en investigación académica y política económica.

### ¿Qué estamos extrayendo?
Extraemos las dos series de precio de trigo que publica el Banco Mundial:

| Variable         | Descripción                                              |
|------------------|----------------------------------------------------------|
| Wheat, US HRW    | Hard Red Winter, precio FOB Gulf, $/tonelada métrica     |
| Wheat, US SRW    | Soft Red Winter, precio FOB Gulf, $/tonelada métrica     |

Estas series son la base para:
- Construir la serie histórica larga de precio (1960–presente) para el EDA
- Deflactar con CPI para obtener precios reales constantes base 2020
- Calcular el spread HRW–SRW como feature del modelo DS-1
- Identificar y documentar los eventos extremos (2008, 2011, 2022)

### Período y granularidad
- Mensual desde enero 1960 hasta el mes más reciente
- ~780 observaciones por serie
- Precios en USD nominales corrientes por tonelada métrica

In [18]:
import pandas as pd
import requests, io

# URL del Pink Sheet más reciente (actualizar según mes)
url = ("https://thedocs.worldbank.org/en/doc/"
       "18675f1d1639c7a34d463f59263ba0a2-0050012025"
       "/world-bank-commodities-price-data-the-pink-sheet")

# Descarga directa del Excel
excel_url = ("https://thedocs.worldbank.org/en/doc/"
             "5d903e848db1d1b83e0ec8f744e55570-0350012021"
             "/CMO-Historical-Data-Monthly.xlsx")

resp = requests.get(excel_url, timeout=30)
xl = pd.ExcelFile(io.BytesIO(resp.content))
print("Hojas disponibles:", xl.sheet_names)

# Hoja de precios mensuales
df = pd.read_excel(
    io.BytesIO(resp.content),
    sheet_name='Monthly Prices',
    skiprows=4,
    header=0,
    index_col=0
)
df.index = pd.to_datetime(df.index, errors='coerce')
df = df.dropna(how='all')

# Filtrar columnas de trigo
wheat_cols = [c for c in df.columns
              if 'Wheat' in str(c) or 'wheat' in str(c)]
print(f"Columnas de trigo encontradas: {wheat_cols}")
df_wheat = df[wheat_cols].dropna(how='all')
print(df_wheat.tail(6))

ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [19]:
import pandas as pd
import requests
import io

# ── URL CORRECTA (con /related/ en la ruta) ──────────────────────
# La URL vieja sin /related/ devuelve HTML → pandas lanza ValueError
EXCEL_URL = (
    "https://thedocs.worldbank.org/en/doc/"
    "18675f1d1639c7a34d463f59263ba0a2-0050012025"
    "/related/CMO-Historical-Data-Monthly.xlsx"  # 👈 /related/ es clave
)

# Headers para evitar bloqueo del servidor
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/120.0.0.0"
    )
}

# ── DESCARGA ─────────────────────────────────────────────────────
resp = requests.get(EXCEL_URL, headers=headers, timeout=60)

# Verificar que realmente es un Excel (empieza con PK = ZIP header)
magic = resp.content[:4].hex()
if not magic.startswith("504b"):  # 504b = PK (ZIP/XLSX)
    print(f"⚠️  La URL devolvió HTML, no Excel. Magic bytes: {magic}")
    print("   Primeros 200 chars:", resp.text[:200])
    raise ValueError("La URL no devuelve un archivo Excel válido.")

print(f"✓ Descargado correctamente: {len(resp.content):,} bytes")

# ── EXPLORAR HOJAS DISPONIBLES ───────────────────────────────────
xl_bytes = io.BytesIO(resp.content)
xl = pd.ExcelFile(xl_bytes, engine="openpyxl")
print("\nHojas disponibles:", xl.sheet_names)

✓ Descargado correctamente: 778,415 bytes

Hojas disponibles: ['AFOSHEET', 'Monthly Prices', 'Monthly Indices', 'Description', 'Index Weights']


In [20]:
# ── LEER HOJA MONTHLY PRICES ─────────────────────────────────────
# OJO: las primeras 4 filas son metadata del WB, la fila 5 es header
xl_bytes = io.BytesIO(resp.content)  # rebobinar buffer

df_raw = pd.read_excel(
    xl_bytes,
    sheet_name="Monthly Prices",
    engine="openpyxl",
    skiprows=4,       # salta las 4 filas de metadata del WB
    header=0,         # fila 5 del Excel = nombres de columnas
    index_col=0       # primera columna = fecha
)

# ── LIMPIAR ÍNDICE DE FECHAS ──────────────────────────────────────
df_raw.index = pd.to_datetime(df_raw.index, errors="coerce")
df_raw = df_raw[df_raw.index.notna()]   # eliminar filas sin fecha
df_raw = df_raw.dropna(how="all")

print(f"\nShape total: {df_raw.shape}")
print(f"Período: {df_raw.index.min().date()} → {df_raw.index.max().date()}")
print(f"\nPrimeras 5 columnas: {df_raw.columns[:5].tolist()}")

# ── FILTRAR COLUMNAS DE TRIGO ────────────────────────────────────
# En el WB el nombre exacto es "Wheat, US HRW" y "Wheat, US SRW"
wheat_cols = [c for c in df_raw.columns 
              if "heat" in str(c)]   # captura Wheat y wheat

print(f"\nColumnas de trigo encontradas ({len(wheat_cols)}):")
for c in wheat_cols:
    print(f"  → '{c}'")

df_wheat = df_raw[wheat_cols].copy()
df_wheat = df_wheat.dropna(how="all")

print(f"\nShape datos trigo: {df_wheat.shape}")
print("\n── ÚLTIMOS 6 MESES ──")
print(df_wheat.tail(6).round(2).to_string())

# ── GUARDAR CSV ──────────────────────────────────────────────────
df_wheat.to_csv("wb_wheat_prices.csv")
print("\n✓ Guardado: wb_wheat_prices.csv")


Shape total: (0, 71)
Período: NaT → NaT

Primeras 5 columnas: ['Crude oil, average', 'Crude oil, Brent', 'Crude oil, Dubai', 'Crude oil, WTI', 'Coal, Australian']

Columnas de trigo encontradas (2):
  → 'Wheat, US SRW'
  → 'Wheat, US HRW'

Shape datos trigo: (0, 2)

── ÚLTIMOS 6 MESES ──
Empty DataFrame
Columns: [Wheat, US SRW, Wheat, US HRW]
Index: []

✓ Guardado: wb_wheat_prices.csv


C:\Users\carlo\AppData\Local\Temp\ipykernel_30132\144463047.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_raw.index = pd.to_datetime(df_raw.index, errors="coerce")


In [21]:
import pandas as pd
import requests
import io

EXCEL_URL = (
    "https://thedocs.worldbank.org/en/doc/"
    "18675f1d1639c7a34d463f59263ba0a2-0050012025"
    "/related/CMO-Historical-Data-Monthly.xlsx"
)

headers = {"User-Agent": "Mozilla/5.0 Chrome/120.0.0.0"}
resp = requests.get(EXCEL_URL, headers=headers, timeout=60)
xl_bytes = io.BytesIO(resp.content)

# ── PASO 1: ver las primeras 15 filas SIN skiprows ────────────────
df_inspect = pd.read_excel(
    xl_bytes,
    sheet_name="Monthly Prices",
    engine="openpyxl",
    header=None,    # sin asumir nada
    nrows=15        # solo las primeras 15 filas
)

print("=== ESTRUCTURA CRUDA DEL ARCHIVO ===")
print(df_inspect.to_string())

=== ESTRUCTURA CRUDA DEL ARCHIVO ===
                                                           0                   1                 2                 3               4                 5                       6                7                    8                             9                  10      11               12               13                   14            15            16            17           18          19         20                21        22               23        24           25            26            27             28         29      30       31              32               33              34                    35             36             37              38          39      40       41          42       43                44         45         46            47                       48               49               50                  51                   52             53               54                55            56              57      58      59    

## Fuente 5: CBOT Wheat Futures — Precios Diarios (yfinance / Stooq)
**Símbolo:** ZW=F (yfinance) | ZW.F (Stooq)  
**Acceso:** Gratuito, sin registro  
**Actualización:** Diaria (días hábiles de mercado)  

### ¿Qué es?
El contrato de futuros de trigo del Chicago Board of Trade (CBOT), símbolo ZW, es el 
instrumento financiero de referencia global para el precio del trigo. Es el precio que 
los traders, exportadores e importadores de todo el mundo usan para cubrir riesgo de precio 
(hedging) y especular sobre el mercado triguero. Cotiza en US cents por bushel y se liquida 
en 5 fechas por año (marzo, mayo, julio, septiembre, diciembre). Es la señal de precio 
más líquida y actualizada disponible para el mercado de trigo.

### ¿Qué estamos extrayendo?
Descargamos la serie histórica diaria del contrato front-month (el más próximo a vencer) 
desde el año 2000, convirtiendo las unidades a USD/tonelada métrica para homologar con 
las demás fuentes:

| Variable          | Descripción                                          |
|-------------------|------------------------------------------------------|
| close_cents_bu    | Precio de cierre en US cents/bushel (unidad original)|
| price_usd_mt      | Precio convertido a USD/tonelada métrica             |
| volume            | Volumen de contratos negociados                      |
| ma_20d / ma_60d   | Medias móviles 20 y 60 días (features DS-1)          |
| vol_20d           | Volatilidad realizada 20 días (feature DS-1)         |
| price_lag_4w      | Precio rezagado 4 semanas (autocorrelación)          |

**Conversión de unidades:**  
`precio_usd_mt = (cents_por_bushel / 100) × 36.7437`  
*(1 tonelada métrica = 36.7437 bushels de trigo)*

### Período y granularidad
- Diario desde enero 2000 (~6,500 observaciones)
- Solo días hábiles de mercado (lunes a viernes)
- Fuente más granular del proyecto — base para el modelo DS-1

In [23]:
import pandas as pd
import yfinance as yf

# Opción A: yfinance (más estable)
# ZW=F = Wheat Futures front month CBOT
ticker = yf.Ticker("ZW=F")
df_raw = ticker.history(start="2000-01-01", interval="1d")

# Conversión: cents/bushel → USD/tonelada métrica
# 1 bushel trigo = 27.2155 kg → 1 TM = 36.74 bushels
df_raw['price_usdmt'] = df_raw['Close'] / 100 * 36.7437

df = df_raw[['Close', 'Volume', 'price_usdmt']].copy()
df.index = pd.to_datetime(df.index).tz_localize(None)
df.columns = ['close_cents_bu', 'volume', 'price_usd_mt']
print(df.tail(5))

# Opción B: pandas_datareader + Stooq (descarga CSV)
# from pandas_datareader import data as pdr
# df_stooq = pdr.get_data_stooq('ZW.F',
#            start='2000-01-01', end='2025-01-01')

# ── Feature engineering básico ──
df['return_1d']     = df['price_usd_mt'].pct_change()
df['ma_20d']        = df['price_usd_mt'].rolling(20).mean()
df['ma_60d']        = df['price_usd_mt'].rolling(60).mean()
df['vol_20d']       = df['return_1d'].rolling(20).std() * (252**0.5)
df['price_lag_4w']  = df['price_usd_mt'].shift(20)  # ~4 semanas
print(df[['price_usd_mt','ma_20d','vol_20d']].tail(5))

            close_cents_bu  volume  price_usd_mt
Date                                            
2026-04-08          580.25  103575    213.205319
2026-04-09          574.50  103129    211.092556
2026-04-10          571.00   84547    209.806527
2026-04-13          582.25   84547    213.940193
2026-04-14          601.75    1195    221.105215
            price_usd_mt      ma_20d   vol_20d
Date                                          
2026-04-08    213.205319  219.943195  0.305944
2026-04-09    211.092556  219.685989  0.306743
2026-04-10    209.806527  219.295588  0.305946
2026-04-13    213.940193  218.629608  0.266865
2026-04-14    221.105215  218.712281  0.269385


## Fuente 6: Agriculture Canada — Outlook for Principal Field Crops
**URL:** https://agriculture.canada.ca/en/sector/crops/reports-statistics  
**Acceso:** PDF público sin registro  
**Actualización:** Mensual (aproximadamente el día 18 de cada mes)  

### ¿Qué es?
Agriculture and Agri-Food Canada (AAFC) publica mensualmente su outlook de principales 
cultivos para Canadá, que incluye estimados de producción, área sembrada, exportaciones, 
stocks y precios para el año agrícola en curso y proyecciones para el siguiente. Canadá 
es el segundo exportador mundial de trigo y el primer exportador de trigo de alta proteína 
(Hard Red Spring), por lo que sus proyecciones de cosecha impactan directamente el precio 
CIF puesto en Guatemala.

### ¿Qué estamos extrayendo?
Parseamos las tablas de los PDFs mensuales para extraer:

| Variable                   | Descripción                                      |
|----------------------------|--------------------------------------------------|
| Área sembrada (000 ha)     | Superficie destinada a trigo en Canadá           |
| Producción estimada (000 MT)| Cosecha proyectada del año agrícola              |
| Exportaciones proyectadas  | Volumen de trigo canadiense disponible al mercado|
| Precio promedio farm gate  | Precio en dólares canadienses por bushel         |
| Stocks finales proyectados | Indicador de holgura o tensión de la oferta      |

También extraemos las proyecciones a 2 años (año en curso y siguiente), lo que permite 
alimentar el modelo DS-3 con señales de oferta futura anticipada.

### Período y granularidad
- Reportes mensuales disponibles desde 2015 en PDF
- Datos históricos dentro de cada reporte desde 2020
- Marketing year canadiense: agosto–julio
- Requiere parseo con pdfplumber (tablas complejas con celdas fusionadas)

In [24]:
import pdfplumber
import pandas as pd
import requests, io, re

# URL del reporte más reciente
url = ("https://agriculture.canada.ca/sites/default/files/"
       "documents/2026-01/"
       "Canada%20Outlook%20for%20Principal%20Field%20Crops_202601.pdf")

resp = requests.get(url, timeout=30)
pdf_bytes = io.BytesIO(resp.content)

# Extracción de todas las tablas del PDF
all_tables = []
with pdfplumber.open(pdf_bytes) as pdf:
    print(f"Total páginas: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        for t in tables:
            if t and len(t) > 2:
                df_t = pd.DataFrame(t[1:], columns=t[0])
                df_t['source_page'] = i + 1
                all_tables.append(df_t)
                print(f"  Pág {i+1}: tabla {df_t.shape}")

# Tabla principal: producción de trigo
# Buscar tabla que contenga 'Wheat' o 'Production'
for df_t in all_tables:
    cols_str = ' '.join([str(c) for c in df_t.columns])
    if 'Wheat' in cols_str or 'Production' in cols_str:
        print("\n── Tabla de producción encontrada ──")
        print(df_t.to_string())
        break

# Limpiar y convertir valores numéricos
def clean_num(x):
    if pd.isna(x): return None
    return float(str(x).replace(',','').replace(' ','')
                         .strip()) if x else None

Total páginas: 13
  Pág 1: tabla (20, 31)
  Pág 1: tabla (2, 2)


## Fuente 7: FAO GIEWS / WFP VAM — Precios de Alimentos Guatemala
**URL:** https://data.humdata.org/dataset/wfp-food-prices-for-guatemala  
**Acceso:** API pública + CSV descargable sin registro  
**Actualización:** Mensual  

### ¿Qué es?
El Global Information and Early Warning System (GIEWS) de la FAO y el sistema VAM 
(Vulnerability Analysis and Mapping) del Programa Mundial de Alimentos (WFP) monitorean 
los precios de alimentos en mercados locales de más de 90 países en desarrollo. Para 
Guatemala, registran precios de harina de trigo, maíz, frijol y otros alimentos básicos 
en mercados de Ciudad de Guatemala y otras ciudades, permitiendo trazar la cadena de 
transmisión de precios desde el mercado global hasta el consumidor final guatemalteco.

### ¿Qué estamos extrayendo?
Extraemos los precios mensuales de harina de trigo en mercados guatemaltecos:

| Variable       | Descripción                                            |
|----------------|--------------------------------------------------------|
| date           | Mes y año de la observación                            |
| market         | Nombre del mercado (ej. Guatemala City, Quetzaltenango)|
| commodity      | Producto (Wheat flour / Harina de trigo)               |
| price          | Precio en moneda local                                 |
| currency       | GTQ (Quetzales guatemaltecos)                          |
| unit           | Unidad de medida (kg o lb según el mercado)            |

Este dataset es el único que conecta el precio internacional del trigo con el precio 
real pagado en Guatemala, permitiendo calcular el pass-through (transmisión de precios) 
desde CBOT hasta el mercado local — un análisis central del Dashboard 2 del proyecto.

### Período y granularidad
- Mensual desde 2010 hasta el presente
- Múltiples mercados por período (Ciudad de Guatemala, mercados regionales)
- Los precios en GTQ requieren conversión a USD para comparar con CBOT

In [26]:
import pandas as pd
import requests
import io

# ── DIAGNÓSTICO PRIMERO ───────────────────────────────────────────
# Ver qué devuelve la API exactamente
api_url = (
    "https://api.vam.wfp.org/pricetimeseries/Data"
    "?CountryCode=GTM"
    "&CommodityId=71"
    "&MarketId=0"
    "&PriceTypeId=2"
    "&startDate=2010-01-01"
    "&endDate=2026-01-01"
    "&format=json"
)

resp = requests.get(api_url, timeout=30)
print(f"Status code: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type', '')}")
print(f"Primeros 500 chars del response:")
print(resp.text[:500])

Status code: 404
Content-Type: text/html
Primeros 500 chars del response:
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"/>
<title>404 - File or directory not found.</title>
<style type="text/css">
<!--
body{margin:0;font-size:.7em;font-family:Verdana, Arial, Helvetica, sans-serif;background:#EEEEEE;}
fieldset{padding:0 15px 10px 15px;} 
h1{font-size:2.4em;margin:0;color:#FFF;}


In [27]:
import pandas as pd
import requests
import io

df = None  # inicializar

# ══════════════════════════════════════════════════════════════════
# OPCIÓN 1: API WFP VAM — endpoint actualizado 2024
# ══════════════════════════════════════════════════════════════════
# La API vieja (/pricetimeseries/Data) fue deprecada
# El nuevo endpoint es /Markets/PriceMonthly
new_api = (
    "https://api.vam.wfp.org/Markets/PriceMonthly"
    "?iso3=GTM"
    "&commodityName=Wheat+flour"
    "&startDate=2010-01-01"
    "&endDate=2026-01-01"
)

headers = {"Accept": "application/json"}

try:
    resp1 = requests.get(new_api, headers=headers, timeout=30)
    print(f"Opción 1 status: {resp1.status_code}")
    if resp1.status_code == 200:
        data = resp1.json()
        print(f"Keys en response: {list(data.keys()) if isinstance(data, dict) else type(data)}")
        # El response puede ser lista directa o dict con key 'data'
        records = data if isinstance(data, list) else data.get('data', data.get('items', []))
        df = pd.DataFrame(records)
        print(f"Columnas: {df.columns.tolist()}")
        print(df.head(3))
except Exception as e:
    print(f"Opción 1 falló: {e}")

# ══════════════════════════════════════════════════════════════════
# OPCIÓN 2: HDX / Humanitarian Data Exchange — CSV directo
# Dataset oficial WFP para Guatemala, actualizado mensualmente
# ══════════════════════════════════════════════════════════════════
if df is None or len(df) == 0:
    print("\n── Intentando Opción 2: HDX CSV directo ──")
    
    # URL directa del CSV en HDX (WFP Food Prices Guatemala)
    hdx_csv = (
        "https://data.humdata.org/dataset/"
        "ec418834-5e47-4d38-91d1-feafa14c6e95/"
        "resource/4bf28c0e-3b14-4e6e-92d3-dfc8eb2f8eb6/"
        "download/wfp_food_prices_gtm.csv"
    )
    
    try:
        resp2 = requests.get(hdx_csv, timeout=30)
        print(f"Opción 2 status: {resp2.status_code}")
        if resp2.status_code == 200:
            df = pd.read_csv(io.StringIO(resp2.text))
            print(f"Shape: {df.shape}")
            print(f"Columnas: {df.columns.tolist()}")
            print(df.head(3))
    except Exception as e:
        print(f"Opción 2 falló: {e}")

# ══════════════════════════════════════════════════════════════════
# OPCIÓN 3: Descarga manual — instrucciones
# ══════════════════════════════════════════════════════════════════
if df is None or len(df) == 0:
    print("""
── Opción 3: Descarga manual (2 minutos) ──

1. Ve a: https://data.humdata.org/dataset/wfp-food-prices-for-guatemala
2. Busca el archivo: wfp_food_prices_gtm.csv
3. Descárgalo y ponlo en la misma carpeta que este notebook
4. Ejecuta:

   df = pd.read_csv('wfp_food_prices_gtm.csv')
   print(df.head())
""")

Opción 1 status: 404

── Intentando Opción 2: HDX CSV directo ──
Opción 2 status: 404

── Opción 3: Descarga manual (2 minutos) ──

1. Ve a: https://data.humdata.org/dataset/wfp-food-prices-for-guatemala
2. Busca el archivo: wfp_food_prices_gtm.csv
3. Descárgalo y ponlo en la misma carpeta que este notebook
4. Ejecuta:

   df = pd.read_csv('wfp_food_prices_gtm.csv')
   print(df.head())



## Fuente 8: Baltic Dry Index (BDI) — Costo de Flete Marítimo
**Fuente FRED:** Serie DCOILBRENTEU (proxy) | Stooq: BDI.UK  
**Acceso:** Gratuito via FRED API o Stooq sin registro  
**Actualización:** Diaria  

### ¿Qué es?
El Baltic Dry Index (BDI) es publicado diariamente por el Baltic Exchange de Londres y 
mide el costo de fletar barcos de carga seca a granel (buques Capesize, Panamax y 
Supramax) en las principales rutas marítimas del mundo. Es el indicador de referencia 
del costo de transporte marítimo de commodities como trigo, maíz, carbón y mineral de 
hierro. Para Guatemala, que importa todo su trigo por vía marítima (puertos de 
Puerto Quetzal en el Pacífico y Puerto Barrios en el Atlántico), el BDI impacta 
directamente el precio CIF final del trigo importado.

### ¿Qué estamos extrayendo?
Extraemos la serie mensual del BDI y construimos features de rezago para el modelo DS-1:

| Variable       | Descripción                                              |
|----------------|----------------------------------------------------------|
| bdi            | Índice mensual promedio (puntos)                         |
| bdi_lag1m      | BDI rezagado 1 mes                                       |
| bdi_lag2m      | BDI rezagado 2 meses                                     |
| bdi_ma3m       | Media móvil 3 meses (suaviza estacionalidad)             |
| bdi_yoy        | Variación año contra año (%)                             |

La correlación esperada entre BDI y precio CIF trigo en Guatemala es de +0.35 a +0.50 
con un rezago de 4 a 6 semanas, ya que el costo de flete se negocia antes de la carga 
y se refleja en el precio de llegada semanas después.

### Período y granularidad
- Diario desde 1985 (serie Stooq) / Mensual desde 1985 (serie FRED)
- Para el proyecto se usa frecuencia mensual resampleada
- Outliers documentados: colapso 2008–2009, mínimo histórico 2016, COVID 2020

In [35]:
import pandas as pd
from fredapi import Fred

fred = Fred(api_key='2f55df367c8ec1c44c28ed52b959e6a5')

# ── Explorar qué series de flete tiene FRED disponibles ──────────
series_flete = {
    'PCU483111483111': 'Deep Sea Freight Transportation PPI',
    'PCU4831114831111': 'Deep Sea Foreign Freight PPI',
    'PPIACO':          'PPI All Commodities',
    'WPU3012':         'PPI Industrial Chemicals (proxy logístico)',
}

disponibles = {}
for sid, desc in series_flete.items():
    try:
        s = fred.get_series(sid, observation_start='2000-01-01')
        print(f"✓ {sid} — {desc}")
        print(f"  Registros: {len(s)} | Último: {s.iloc[-1]:.2f} ({s.index[-1].date()})\n")
        disponibles[sid] = s
    except Exception as e:
        print(f"✗ {sid} — {e}\n")

✓ PCU483111483111 — Deep Sea Freight Transportation PPI
  Registros: 315 | Último: 409.16 (2026-03-01)

✗ PCU4831114831111 — Bad Request.  The series does not exist.

✓ PPIACO — PPI All Commodities
  Registros: 315 | Último: 274.10 (2026-03-01)

✓ WPU3012 — PPI Industrial Chemicals (proxy logístico)
  Registros: 202 | Último: 159.52 (2026-03-01)



In [37]:
import pandas as pd
from fredapi import Fred

fred = Fred(api_key='2f55df367c8ec1c44c28ed52b959e6a5')

# ── Descargar PCU483111483111 — Deep Sea Freight Transportation PPI
# Es el proxy oficial de costo de flete marítimo disponible en FRED
# Publicado por el Bureau of Labor Statistics (BLS) — mensual
raw = fred.get_series('PCU483111483111', observation_start='2000-01-01')
raw.index = pd.to_datetime(raw.index)
raw = raw.sort_index()
raw.name = 'bdi_proxy'

print(f"✓ Serie descargada")
print(f"  Registros : {len(raw)}")
print(f"  Período   : {raw.index.min().date()} → {raw.index.max().date()}")
print(f"  Min       : {raw.min():.2f}")
print(f"  Max       : {raw.max():.2f}")
print(f"  Promedio  : {raw.mean():.2f}")
print(f"  Último    : {raw.iloc[-1]:.2f} ({raw.index[-1].date()})")

✓ Serie descargada
  Registros : 315
  Período   : 2000-01-01 → 2026-03-01
  Min       : 141.70
  Max       : 468.16
  Promedio  : 272.54
  Último    : 409.16 (2026-03-01)


In [38]:
# ── Construir features para el modelo DS-1 ───────────────────────
df_bdi_m = raw.resample('MS').mean()  # asegurar frecuencia mensual
df_bdi_m.name = 'bdi'

df_bdi_features = pd.DataFrame({
    'bdi':       df_bdi_m,
    'bdi_lag1m': df_bdi_m.shift(1),
    'bdi_lag2m': df_bdi_m.shift(2),
    'bdi_ma3m':  df_bdi_m.rolling(3).mean(),
    'bdi_yoy':   df_bdi_m.pct_change(12).round(4)
}).dropna()

print(f"\nShape features: {df_bdi_features.shape}")
print(f"\n── Últimos 6 meses ──")
print(df_bdi_features.tail(6).round(2).to_string())


Shape features: (303, 5)

── Últimos 6 meses ──
               bdi  bdi_lag1m  bdi_lag2m  bdi_ma3m  bdi_yoy
2025-10-01  420.70     390.95     392.83    401.49     0.03
2025-11-01  402.72     420.70     390.95    404.79     0.02
2025-12-01  415.30     402.72     420.70    412.91     0.05
2026-01-01  445.27     415.30     402.72    421.10     0.04
2026-02-01  409.88     445.27     415.30    423.48    -0.06
2026-03-01  409.16     409.88     445.27    421.44    -0.02


In [39]:
# ── Estadísticas descriptivas completas ──────────────────────────
print("\n── Estadísticas descriptivas ──")
print(df_bdi_features.describe().round(2).to_string())

# ── Eventos extremos documentados en la serie ────────────────────
print("\n── Top 5 meses con flete MÁS ALTO ──")
print(df_bdi_features['bdi']
      .nlargest(5)
      .reset_index()
      .rename(columns={'index':'fecha', 'bdi':'valor'})
      .to_string(index=False))

print("\n── Top 5 meses con flete MÁS BAJO ──")
print(df_bdi_features['bdi']
      .nsmallest(5)
      .reset_index()
      .rename(columns={'index':'fecha', 'bdi':'valor'})
      .to_string(index=False))


── Estadísticas descriptivas ──
          bdi  bdi_lag1m  bdi_lag2m  bdi_ma3m  bdi_yoy
count  303.00     303.00     303.00    303.00   303.00
mean   277.17     276.34     275.51    276.34     0.04
std     72.85      72.76      72.67     72.51     0.10
min    158.60     158.60     158.50    160.27    -0.25
25%    231.80     231.60     231.40    231.95    -0.02
50%    253.40     252.90     252.80    252.90     0.03
75%    303.50     302.50     301.60    302.82     0.11
max    468.16     468.16     468.16    466.06     0.37

── Top 5 meses con flete MÁS ALTO ──
     fecha   valor
2022-09-01 468.156
2022-10-01 468.110
2022-11-01 461.904
2022-12-01 459.372
2022-08-01 458.815

── Top 5 meses con flete MÁS BAJO ──
     fecha  valor
2001-04-01  158.6
2001-01-01  162.5
2001-03-01  162.8
2001-02-01  164.2
2002-04-01  167.6


In [40]:
# ── Guardar ───────────────────────────────────────────────────────
df_bdi_features.to_csv('bdi_monthly_features.csv')
print("\n✓ Guardado: bdi_monthly_features.csv")
print(f"  Columnas: {df_bdi_features.columns.tolist()}")


✓ Guardado: bdi_monthly_features.csv
  Columnas: ['bdi', 'bdi_lag1m', 'bdi_lag2m', 'bdi_ma3m', 'bdi_yoy']
